# Notebook 11: Cross-fitted F1 repair — development pilot

Select **T4 GPU**, then run the **single executable cell**. Your existing project and Notebook 09 outputs are read automatically. Keep the original folder names. No downloads or manual uploads are needed.

This is a new experiment, not an assurance of higher scores or established novelty. The controller's original training used only one DSADS policy participant and three HAR policy participants. This pilot expands training outcomes to the original **train + policy** participants using out-of-subject auxiliary models. It compares accuracy-based utility, exact scaled single-replacement Macro-F1 utility, and mean/minimum utility across three source-participant jackknife heads. The minimum is a disagreement rejection heuristic, not a certified risk bound or formal distributionally robust optimizer. Single-replacement F1 utilities do not guarantee improvement when many repairs are made together.

**Compute:** two configurations only: fold 0, seed 2026 on both datasets. It adds four DSADS and five HAR classifier/reconstructor pairs, using the previous small architectures with fixed 30/25 epochs, plus small CPU controllers. This is substantially more work than Notebook 09. Actual runtime depends on Colab availability. A 75-minute session budget saves progress and pauses between epochs, conditions and head fits. Rerun the same cell to resume. Do not change settings midway. Existing finished work is reused; old outputs are never overwritten. GPU is required only while auxiliary fits/caches remain unfinished.

**Separation:** auxiliary normalization is fitted on that auxiliary model's training participants only. Held source labels are used only to form controller targets after prediction. The original policy caches are reused as their already out-of-subject cohort; their original model checkpoint selection used validation, as before. New auxiliary fits have fixed final epochs and use no held-cohort or validation checkpoint selection. Original validation candidate predictions are reused for all new policies, and thresholds use validation only. Only clean/dropout/bias/gain are used. No test prediction files are read or created. Prepared whole-dataset arrays are opened to select source rows and verify participant separation; this is not a claim that test labels were never stored or loaded.

This extension is motivated after previous test results were inspected. A successful validation pilot would still need further evaluation, with the old test set described as exploratory. Do not choose separate winning policies for each dataset from test scores.

**Literature positioning:** F-measure optimization is established ([Joachims, ICML 2005](https://www.cs.cornell.edu/~tj/publications/joachims_05a.pdf)), and sensor cleaning plus robust HAR fusion already exists ([Centaur](https://arxiv.org/abs/2303.04636)). The contribution to investigate is this particular combination of out-of-subject action outcomes, participant-balanced marginal F1 targets, and rejection under source-head disagreement. These ingredients and a search without an exact match do not establish novelty by themselves. Compare the mechanism and protocol against the closest work before making a novelty claim.

Outputs: `SensorRepair_ICDSIS/outputs/notebook11_v1`. Share the final output and `notebook11_validation_summary.csv`, `notebook11_run_report.json`, and `notebook11_source_audit.csv`. If it pauses, rerun the same cell until both configurations complete.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
import gc
import hashlib
import json
import math
import random
import shutil
import time
import zlib
import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from threadpoolctl import threadpool_limits

PROJECT=Path('/content/drive/MyDrive/SensorRepair_ICDSIS')
OUTPUT=PROJECT/'outputs'/'notebook11_v1'
LOCAL=Path('/content/sensorrepair_notebook11_v1')
CODE_VERSION='sensorrepair_nb11_v1'
CELL_SIGNATURE='9a8b5597d3268627e231f9819481af36a384eb1b304d3807ba40a1f81feee1a3'
CLASSIFIER_EPOCHS=30
RECONSTRUCTOR_EPOCHS=25
BATCH_SIZE=128
INFERENCE_BATCH_SIZE=128
LEARNING_RATE=.001
MAX_ITER=160
SESSION_MINUTES=75
REQUIRE_GPU=True
HARM_PENALTIES=[1.,2.,4.]
THRESHOLDS=[0.,.005,.01,.02,.05,.1,.25,.5,1.,None]
CLEAN_F1_TOLERANCE=.005
MAX_HARM=.01
FAMILIES=['oof_accuracy_utility','oof_f1_utility','source_mean_f1','source_min_f1']
CONDITIONS=['clean']+[f'{f}_{i}' for f in ['dropout','bias','gain'] for i in [1,2,3]]
JOBS=[{'dataset':d,'fold':'fold_0','seed':2026} for d in ['dsads','har']]
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT.mkdir(parents=True,exist_ok=True)
LOCAL.mkdir(parents=True,exist_ok=True)
torch.set_num_threads(2)
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True
torch.use_deterministic_algorithms(True)
if DEVICE.type=='cuda':torch.backends.cuda.matmul.allow_tf32=False
SESSION_DEADLINE=time.monotonic()+SESSION_MINUTES*60
print('NOTEBOOK 11 | cross-fitted F1 repair with source disagreement rejection | one executable cell',flush=True)
print('Development pilot: both datasets, fold_0, seed 2026. No final test predictions or stuck-fault evaluation.',flush=True)
print('Device:',torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU','| PyTorch:',torch.__version__,flush=True)
print('Auxiliary models train from scratch; each held source cohort is excluded from fitting and normalization.',flush=True)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def json_signature(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()


def save_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(temporary, path)


def save_csv(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    value.to_csv(temporary, index=False)
    os.replace(temporary, path)


def save_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    with temporary.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(temporary, path)


def atomic_joblib(path, value):
    temporary = LOCAL / (Path(path).name + '.tmp')
    joblib.dump(value, temporary, compress=3)
    partial = Path(path).with_name(Path(path).name + '.partial')
    shutil.copyfile(temporary, partial)
    os.replace(partial, path)


def save_checkpoint(path, state):
    path = Path(path)
    temporary = LOCAL / (path.name + '.tmp')
    torch.save(state, temporary)
    partial = path.with_name(path.name + '.partial')
    shutil.copyfile(temporary, partial)
    os.replace(partial, path)


def load_checkpoint(path):
    return torch.load(path, map_location='cpu', weights_only=True)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(seed)


def cuda_sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()


class ActivityCNN(nn.Module):
    def __init__(self, channels, classes):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(channels * 2, 64, 7, padding=3), nn.GroupNorm(8, 64), nn.GELU(),
            nn.Conv1d(64, 96, 5, stride=2, padding=2), nn.GroupNorm(8, 96), nn.GELU(),
            nn.Conv1d(96, 128, 5, stride=2, padding=2), nn.GroupNorm(8, 128), nn.GELU(),
            nn.AvgPool1d(kernel_size=8, stride=8), nn.Flatten(),
            nn.Linear(512, 128), nn.GELU(), nn.Dropout(0.1), nn.Linear(128, classes))

    def forward(self, values, available=None):
        if available is None:
            available = torch.ones_like(values)
        return self.encoder(torch.cat([values, available], dim=1))


class SensorReconstructor(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(channels * 2, 64, 5, padding=2), nn.GELU(),
            nn.Conv1d(64, 96, 5, padding=2), nn.GELU(),
            nn.Conv1d(96, 64, 5, padding=2), nn.GELU(), nn.Conv1d(64, channels, 1))

    def forward(self, values, available):
        return self.network(torch.cat([values, available], dim=1))


def mask_groups(values, groups, rng):
    masked = np.array(values, dtype=np.float32, copy=True)
    available = np.ones_like(masked)
    for i in range(len(masked)):
        channels = groups[int(rng.integers(len(groups)))]['channels']
        masked[i, channels, :] = 0.0
        available[i, channels, :] = 0.0
    return masked, available


def classifier_epoch_inputs(clean, groups, seed):
    rng = np.random.default_rng(seed)
    values = np.array(clean, copy=True)
    available = np.ones_like(values)
    choices = rng.random(len(values))
    faults = np.flatnonzero((choices >= 0.35) & (choices < 0.80))
    masks = np.flatnonzero(choices >= 0.80)
    values[faults] = corrupt_signals(clean[faults], groups, rng)
    masked, indicators = mask_groups(clean[masks], groups, rng)
    values[masks], available[masks] = masked, indicators
    return values, available


def cpu_state(model):
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}


@torch.inference_mode()
def action_outputs(classifier, reconstructor, values, groups, classes):
    classifier.eval()
    reconstructor.eval()
    number = len(values)
    group_count = len(groups)
    probabilities = np.empty((number, 1 + 2 * group_count, classes), dtype=np.float32)
    residuals = np.empty((number, group_count), dtype=np.float32)
    group_statistics = np.empty((number, group_count, 4), dtype=np.float32)
    cuda_sync()
    started = time.perf_counter()
    for start in range(0, number, INFERENCE_BATCH_SIZE):
        stop = min(start + INFERENCE_BATCH_SIZE, number)
        observed = torch.as_tensor(values[start:stop], device=DEVICE)
        probabilities[start:stop, 0] = classifier(observed).softmax(dim=1).cpu().numpy()
        for index, group in enumerate(groups):
            channels = group['channels']
            masked, available = observed.clone(), torch.ones_like(observed)
            masked[:, channels, :], available[:, channels, :] = 0.0, 0.0
            reconstruction = reconstructor(masked, available)
            repaired = observed.clone()
            repaired[:, channels, :] = reconstruction[:, channels, :]
            probabilities[start:stop, 1 + index] = classifier(masked, available).softmax(dim=1).cpu().numpy()
            probabilities[start:stop, 1 + group_count + index] = classifier(repaired).softmax(dim=1).cpu().numpy()
            residuals[start:stop, index] = (observed[:, channels, :] - reconstruction[:, channels, :]).square().mean(dim=(1, 2)).cpu().numpy()
            signal = values[start:stop, channels, :]
            group_statistics[start:stop, index] = np.stack([
                np.abs(signal).mean(axis=(1, 2)), signal.std(axis=(1, 2)),
                np.abs(np.diff(signal, axis=2)).mean(axis=(1, 2)),
                (signal == 0).mean(axis=(1, 2))], axis=1)
    cuda_sync()
    seconds = time.perf_counter() - started
    assert np.isfinite(probabilities).all() and np.isfinite(residuals).all()
    assert np.allclose(probabilities.sum(axis=2), 1.0, atol=1e-5)
    return probabilities, residuals, group_statistics, seconds


def corrupt_signals(values, groups, rng, fault=None, severity=None):
    # Fault locations and severity metadata are never returned as model features.
    result = np.array(values, dtype=np.float32, copy=True)
    for i in range(len(result)):
        kind = fault if fault is not None else str(rng.choice(['dropout', 'bias', 'gain']))
        level = int(severity) if severity is not None else int(rng.integers(0, 3))
        group = groups[int(rng.integers(len(groups)))]['channels']
        if kind == 'dropout':
            fraction = protocol['fault_severities']['dropout']['time_fraction'][level]
            length = max(1, int(round(result.shape[2] * fraction)))
            start = int(rng.integers(0, result.shape[2] - length + 1))
            result[i, group, start:start + length] = 0.0
        elif kind == 'bias':
            amount = protocol['fault_severities']['bias']['absolute_offset_in_training_std'][level]
            signs = rng.choice([-1.0, 1.0], size=len(group)).astype(np.float32)
            result[i, group, :] += np.float32(amount) * signs[:, None]
        elif kind == 'gain':
            factor = float(protocol['fault_severities']['gain']['factor'][level])
            factor = factor if rng.random() < 0.5 else 1.0 / factor
            result[i, group, :] *= np.float32(factor)
        elif kind == 'stuck':
            fraction = protocol['fault_severities']['stuck']['time_fraction'][level]
            length = max(1, int(round(result.shape[2] * fraction)))
            start = int(rng.integers(0, result.shape[2] - length + 1))
            first = result[i, group, start].copy()
            result[i, group, start:start + length] = first[:, None]
        else:
            raise ValueError(f'Unknown fault: {kind}')
    return result


def evaluate_choices(pack, choices, method):
    p, y = pack['probabilities'], pack['y']
    D, N, A, K = p.shape
    assert choices.shape == (D, N) and choices.min() >= 0 and choices.max() < A
    predictions = p.argmax(axis=-1)
    original_correct = predictions[:, :, 0] == y[None, :]
    selected = np.take_along_axis(predictions, choices[:, :, None], axis=2).squeeze(2)
    rows = []
    for i, condition in enumerate(pack['conditions']):
        correct = selected[i] == y
        helpful = float((~original_correct[i] & correct).mean())
        harmful = float((original_correct[i] & ~correct).mean())
        accuracy = float(correct.mean())
        assert abs(accuracy - original_correct[i].mean() - helpful + harmful) < 1e-10
        rows.append({'condition': condition, 'method': method, 'windows': N,
                     'macro_f1': macro_f1(y, selected[i], K), 'accuracy': accuracy,
                     'intervention_rate': float((choices[i] != 0).mean()), 'helpful_rate': helpful,
                     'harmful_rate': harmful,
                     'harm_given_original_correct': float(np.sum(original_correct[i] & ~correct) / max(1, original_correct[i].sum()))})
    return rows


def aggregate_rows(rows):
    clean = next(row for row in rows if row['condition'] == 'clean')
    corrupted = [row for row in rows if row['condition'] != 'clean']
    answer = {key: float(np.mean([row[key] for row in corrupted])) for key in
              ['macro_f1', 'accuracy', 'intervention_rate', 'helpful_rate', 'harmful_rate']}
    answer.update(clean_macro_f1=clean['macro_f1'], clean_accuracy=clean['accuracy'],
                  clean_harmful_rate=clean['harmful_rate'], clean_intervention_rate=clean['intervention_rate'])
    return answer


class PauseStudy(Exception):
    pass


def check_budget():
    if time.monotonic() >= SESSION_DEADLINE:
        raise PauseStudy('Session time budget reached; all completed epochs and conditions are saved.')

def probability_statistics(probability):
    sorted_p = np.sort(probability, axis=-1)
    confidence = sorted_p[..., -1]
    margin = confidence - sorted_p[..., -2]
    entropy = -(probability * np.log(np.clip(probability, 1e-7, 1.0))).sum(axis=-1)
    return np.stack([confidence, margin, entropy], axis=-1)


def policy_features(probabilities, residuals, statistics, residual_scale):
    # This function accepts observable arrays only. It cannot access labels, subjects,
    # source IDs, injected fault locations, or condition/severity metadata.
    D, N, A, K = probabilities.shape
    G, J, M = residuals.shape[-1], A - 1, D * N
    p = probabilities.reshape(M, A, K)
    r = residuals.reshape(M, G)
    s = statistics.reshape(M, G, 4)
    original, candidates = p[:, 0], p[:, 1:]
    candidate_groups = np.tile(np.arange(G), 2)
    original_statistics = probability_statistics(original)
    candidate_statistics = probability_statistics(candidates)
    scaled = r / residual_scale[None, :]

    def repeat(values):
        return np.broadcast_to(values[:, None, :], (M, J, values.shape[-1]))

    middle = (original[:, None, :] + candidates) / 2.0
    js = 0.5 * (original[:, None, :] * np.log(np.clip(original[:, None, :], 1e-7, 1.0) / np.clip(middle, 1e-7, 1.0))).sum(axis=-1)
    js += 0.5 * (candidates * np.log(np.clip(candidates, 1e-7, 1.0) / np.clip(middle, 1e-7, 1.0))).sum(axis=-1)
    names, pieces = [], []

    def add(values, labels):
        assert values.shape == (M, J, len(labels))
        pieces.append(values)
        names.extend(labels)

    add(repeat(original), [f'original_probability_{k}' for k in range(K)])
    add(candidates, [f'action_probability_{k}' for k in range(K)])
    add(candidates - original[:, None, :], [f'probability_difference_{k}' for k in range(K)])
    add(repeat(original_statistics), ['original_confidence', 'original_margin', 'original_entropy'])
    add(candidate_statistics, ['action_confidence', 'action_margin', 'action_entropy'])
    add(candidate_statistics - original_statistics[:, None, :], ['confidence_difference', 'margin_difference', 'entropy_difference'])
    add(js[..., None], ['prediction_js_divergence'])
    add((candidates.argmax(axis=-1) == original.argmax(axis=-1)[:, None])[..., None], ['prediction_agreement'])
    kinds = np.zeros((J, 2), dtype=np.float32)
    kinds[:G, 0], kinds[G:, 1] = 1.0, 1.0
    add(np.broadcast_to(kinds[None], (M, J, 2)), ['action_is_mask', 'action_is_reconstruction'])
    add(np.broadcast_to(np.eye(G, dtype=np.float32)[candidate_groups][None], (M, J, G)), [f'candidate_group_{g}' for g in range(G)])
    add(repeat(np.log1p(r)), [f'log_residual_group_{g}' for g in range(G)])
    add(np.log1p(r[:, candidate_groups])[..., None], ['candidate_log_residual'])
    add(np.log1p(scaled[:, candidate_groups])[..., None], ['candidate_log_scaled_residual'])
    add(repeat(np.log1p(scaled.max(axis=-1, keepdims=True))), ['maximum_log_scaled_residual'])
    add(s[:, candidate_groups, :], ['candidate_mean_absolute_signal', 'candidate_signal_std', 'candidate_temporal_difference', 'candidate_zero_fraction'])
    add(repeat(s.reshape(M, G * 4)), [f'observed_group_{g}_stat_{j}' for g in range(G) for j in range(4)])
    values = np.ascontiguousarray(np.concatenate(pieces, axis=-1).reshape(M * J, -1), dtype=np.float32)
    assert len(set(names)) == len(names) and np.isfinite(values).all()
    return values, names


def macro_f1(y, prediction, classes):
    matrix = np.bincount(y * classes + prediction, minlength=classes * classes).reshape(classes, classes)
    numerator = 2.0 * matrix.diagonal()
    denominator = matrix.sum(axis=0) + matrix.sum(axis=1)
    return float(np.divide(numerator, denominator, out=np.zeros(classes), where=denominator > 0).mean())


def pack_from_caches(paths, expected_signature):
    matrices, residuals, statistics = [], [], []
    anchor, conditions = None, []
    for path in paths:
        with np.load(path, allow_pickle=False) as saved:
            assert saved['signature'].item() == expected_signature
            identity = {key: saved[key] for key in ['y', 'subjects', 'window_ids', 'source_indices', 'action_names']}
            if anchor is None:
                anchor = identity
            else:
                for key, value in identity.items():
                    assert np.array_equal(value, anchor[key]), f'Cache alignment error: {key}'
            matrices.append(saved['probabilities'])
            residuals.append(saved['reconstruction_residuals'])
            statistics.append(saved['observed_group_statistics'])
            conditions.append(str(saved['condition'].item()))
    pack = dict(anchor)
    pack.update(probabilities=np.stack(matrices), residuals=np.stack(residuals),
                statistics=np.stack(statistics), conditions=conditions)
    p, r, s = pack['probabilities'], pack['residuals'], pack['statistics']
    D, N, A, K = p.shape
    G = r.shape[-1]
    assert A == 1 + 2 * G and r.shape == (D, N, G) and s.shape == (D, N, G, 4)
    assert np.isfinite(p).all() and np.isfinite(r).all() and np.isfinite(s).all()
    assert np.all(p >= 0) and np.allclose(p.sum(axis=-1), 1.0, atol=1e-5)
    assert np.all(r >= 0) and len(np.unique(pack['window_ids'])) == N
    return pack


def identity(job):
    return f'{job["dataset"]}_{job["fold"]}_seed_{job["seed"]}'

# Only hashes and file identities are read from previous reports. Previous test
# metrics are never inputs to the pilot's fits, thresholds or validation scores.
PREPARED=PROJECT/'processed/v1'
STAGES={f'nb{i:02d}':PROJECT/'outputs'/f'notebook{i:02d}_v1' for i in [2,4,5]}


def verified_copy(source,relative,expected):
    assert not Path(relative).is_absolute() and '..' not in Path(relative).parts
    target=LOCAL/'inputs'/relative
    target.parent.mkdir(parents=True,exist_ok=True)
    if target.is_file() and file_sha256(target)==expected:return target
    partial=target.with_name(target.name+'.partial')
    shutil.copyfile(source,partial)
    assert file_sha256(partial)==expected,f'Input hash mismatch: {source}'
    os.replace(partial,target)
    return target


def stage_json(number,name):
    root=PROJECT/'outputs'/f'notebook{number:02d}_v1'
    inv={r['filename']:r for r in json.loads((root/f'notebook{number:02d}_artifact_inventory.json').read_text())}
    path=verified_copy(root/name,f'nb{number:02d}/{name}',inv[name]['sha256'])
    return json.loads(path.read_text()),inv[name]['sha256']


prep_inventory_path=PREPARED/'artifact_inventory.json'
prep_inventory={r['filename']:r for r in json.loads(prep_inventory_path.read_text())}
completion=json.loads((PREPARED/'notebook01_complete.json').read_text())
assert completion['ready_for_training']
assert completion['artifact_inventory_sha256']==file_sha256(prep_inventory_path)
assert completion['protocol_sha256']==prep_inventory['protocol.json']['sha256']


def prepared_file(name):
    return verified_copy(PREPARED/name,'prepared/'+name,prep_inventory[name]['sha256'])


protocol=json.loads(prepared_file('protocol.json').read_text())
assert protocol['development_fault_types']==['dropout','bias','gain']
source_manifest,source_manifest_sha=stage_json(9,'notebook09_study_manifest.json')
assert json_signature(source_manifest['plan'])==source_manifest['signature']
PRIOR_REPORTS={};PRIOR_HASHES={}
for number in [5,7,9]:
    PRIOR_REPORTS[number],PRIOR_HASHES[str(number)]=stage_json(number,f'notebook{number:02d}_run_report.json')
    assert PRIOR_REPORTS[number]['study_complete']
assert PRIOR_REPORTS[9]['study_signature']==source_manifest['signature']
CACHE_INPUTS={identity(j):{k:v for k,v in source_manifest['plan']['cache_inputs'][identity(j)].items()
                         if k.split('|')[0] in ['policy','validation']} for j in JOBS}
plan={'code_version':CODE_VERSION,'cell_signature':CELL_SIGNATURE,'jobs':JOBS,
 'prepared_inventory_sha256':file_sha256(prep_inventory_path),
 'source_manifest_sha256':source_manifest_sha,'prior_report_hashes':PRIOR_HASHES,'cache_inputs':CACHE_INPUTS,
 'auxiliary_source_pool':'original train union original policy; original policy cohort reuses frozen cache',
 'auxiliary_cohorts':'each DSADS train participant separately; HAR train participants in five seeded equal blocks',
 'auxiliary_model_selection':'none: scratch models, training-only normalization, fixed final epoch',
 'auxiliary_training':{'classifier_epochs':CLASSIFIER_EPOCHS,'reconstructor_epochs':RECONSTRUCTOR_EPOCHS,
   'batch_size':BATCH_SIZE,'learning_rate':LEARNING_RATE,'weight_decay':.0001,
   'augmentation':'original classifier 35% clean, 45% random development fault, 20% group mask',
   'architecture':'unchanged ActivityCNN and SensorReconstructor from Notebook 05'},
 'controller_families':FAMILIES,'controller_head_iterations':MAX_ITER,
 'controller_objective':'subject-balanced exact single-prediction-replacement Macro-F1 delta scaled by total_weight/window_weight',
 'jackknife':'three disjoint blocks of source people, each head excludes one block; targets recomputed on remaining people',
 'jackknife_limitation':'auxiliary signal models may train on another jackknife block; jackknife is of controller outcome rows, not full pipeline refitting',
 'validation_grid':{'harm_penalty':HARM_PENALTIES,'threshold':THRESHOLDS},
 'validation_guards':{'clean_f1_loss':CLEAN_F1_TOLERANCE,'clean_and_mean_corrupt_harm':MAX_HARM},
 'test_predictions_read_or_created':0,'held_out_faults_used':False,
 'status':'development extension after inspection of previous test outcomes; validation is reused and exploratory',
 'software':{'torch':str(torch.__version__),'numpy':str(np.__version__),'sklearn':str(sklearn.__version__)}}
STUDY_SIGNATURE=json_signature(plan)
manifest=OUTPUT/'notebook11_study_manifest.json'
if manifest.exists():
    assert json.loads(manifest.read_text())['signature']==STUDY_SIGNATURE,'Study inputs, settings or environment changed; keep the original configuration.'
else:save_json(manifest,{'signature':STUDY_SIGNATURE,'plan':plan})
print('Development plan frozen:',STUDY_SIGNATURE[:16],flush=True)


def load_original_pack(job,split):
    assert split in ['policy','validation'], 'This pilot does not open test predictions.'
    paths=[];signature=None
    for condition in CONDITIONS:
        item=CACHE_INPUTS[identity(job)][split+'|'+condition]
        if signature is None:signature=item['signature']
        assert signature==item['signature']
        path=verified_copy(STAGES[item['stage']]/item['filename'],item['stage']+'/'+item['filename'],item['sha256'])
        with np.load(path,allow_pickle=False) as z:
            assert z['condition'].item()==condition and z['split'].item()==split
        paths.append(path)
    pack=pack_from_caches(paths,signature)
    assert pack['conditions']==CONDITIONS
    return pack


def load_source_data(job):
    dataset=job['dataset'];fold=job['fold']
    with np.load(prepared_file(dataset+'_signals.npz'),allow_pickle=False) as z:
        data={k:z[k] for k in ['X','y','subjects','window_ids']}
    with np.load(prepared_file(dataset+'_splits.npz'),allow_pickle=False) as z:
        splits={s:z[fold+'_'+s] for s in ['train','policy','validation','test']}
    all_rows=np.concatenate(list(splits.values()))
    assert np.array_equal(np.sort(all_rows),np.arange(len(data['y'])))
    for s,a in splits.items():
        for t,b in splits.items():
            if s!=t:assert not np.intersect1d(data['subjects'][a],data['subjects'][b]).size
    data['splits']=splits
    data['pool']=np.concatenate([splits['train'],splits['policy']])
    data['groups']=protocol['datasets'][dataset]['sensor_groups']
    data['classes']=len(protocol['datasets'][dataset]['class_names'])
    people=np.unique(data['subjects'][splits['train']])
    rng=np.random.default_rng(job['seed']+zlib.crc32((dataset+'|cohorts').encode()))
    people=rng.permutation(people)
    blocks=[b.tolist() for b in (np.array_split(people,len(people)) if dataset=='dsads' else np.array_split(people,5))]
    assert all(len(b)>0 for b in blocks)
    data['blocks']=blocks
    return data


def fit_auxiliary(kind,clean,y,groups,classes,run,signature,stage_seed):
    maximum=CLASSIFIER_EPOCHS if kind=='classifier' else RECONSTRUCTOR_EPOCHS
    seed_everything(stage_seed)
    model=(ActivityCNN(clean.shape[1],classes) if kind=='classifier' else SensorReconstructor(clean.shape[1])).to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=.0001)
    path=run/(kind+'_last.pt');history=[];first=1
    if path.exists():
        cp=load_checkpoint(path)
        assert cp['signature']==signature and cp['maximum_epochs']==maximum
        model.load_state_dict(cp['model']);optimizer.load_state_dict(cp['optimizer'])
        history=cp['history'];first=cp['epoch']+1
        if first>maximum:
            print(f'  {kind}: reused fixed epoch {maximum}.',flush=True)
            return model
        torch.set_rng_state(cp['torch_rng'])
        if DEVICE.type=='cuda' and cp['cuda_rng']:torch.cuda.set_rng_state_all(cp['cuda_rng'])
        print(f'  {kind}: resuming epoch {first}.',flush=True)
    weights=1./np.sqrt(np.maximum(np.bincount(y,minlength=classes),1))
    weights=torch.tensor(weights/weights.mean(),dtype=torch.float32,device=DEVICE)
    for epoch in range(first,maximum+1):
        check_budget();started=time.perf_counter()
        lr=LEARNING_RATE*(.1+.9*.5*(1.+math.cos(math.pi*(epoch-1)/maximum)))
        for g in optimizer.param_groups:g['lr']=lr
        if kind=='classifier':
            values,available=classifier_epoch_inputs(clean,groups,stage_seed+epoch*1000)
            target=torch.from_numpy(y.astype(np.int64,copy=False))
        else:
            values,available=mask_groups(clean,groups,np.random.default_rng(stage_seed+epoch*1000))
            target=torch.from_numpy(clean)
        loader=DataLoader(TensorDataset(torch.from_numpy(values),torch.from_numpy(available),target),
            batch_size=BATCH_SIZE,shuffle=True,generator=torch.Generator().manual_seed(stage_seed+epoch),
            num_workers=0,pin_memory=DEVICE.type=='cuda')
        model.train();total=0.;seen=0
        for xb,mb,yb in loader:
            xb,mb,yb=xb.to(DEVICE),mb.to(DEVICE),yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True);out=model(xb,mb)
            if kind=='classifier':
                ew=weights[yb];loss=-(out.log_softmax(1).gather(1,yb[:,None]).squeeze(1)*ew).sum()/ew.sum()
            else:
                missing=1.-mb;loss=((out-yb).square()*missing).sum()/missing.sum().clamp_min(1.)
            assert torch.isfinite(loss),'Non-finite auxiliary loss.'
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);optimizer.step()
            total+=float(loss.item())*len(xb);seen+=len(xb)
        cuda_sync()
        history.append({'epoch':epoch,'loss':total/seen,'seconds':time.perf_counter()-started,'learning_rate':lr})
        save_checkpoint(path,{'signature':signature,'maximum_epochs':maximum,'epoch':epoch,'model':cpu_state(model),
            'optimizer':optimizer.state_dict(),'torch_rng':torch.get_rng_state(),
            'cuda_rng':torch.cuda.get_rng_state_all() if DEVICE.type=='cuda' else [],'history':history})
        save_csv(run/(kind+'_history.csv'),pd.DataFrame(history))
        if epoch==1 or epoch%5==0 or epoch==maximum:
            print(f'  {kind} epoch {epoch:02d}/{maximum} | training loss {total/seen:.4f} | no validation selection',flush=True)
        del values,available,target,loader
    return model


def cache_valid(path,signature):
    digest=path.with_name(path.name+'.sha256.json')
    if not(path.exists() and digest.exists()):return False
    meta=json.loads(digest.read_text())
    assert meta['signature']==signature and meta['sha256']==file_sha256(path),'Saved auxiliary artifact changed.'
    return True


def auxiliary_cohort(job,data,number,people,run):
    folder=run/'auxiliary'/f'cohort_{number:02d}';folder.mkdir(parents=True,exist_ok=True)
    held=data['pool'][np.isin(data['subjects'][data['pool']],people)]
    train=data['pool'][~np.isin(data['subjects'][data['pool']],people)]
    tr_people=np.unique(data['subjects'][train]).tolist()
    assert not set(tr_people)&set(people)
    assert not np.intersect1d(np.concatenate([train,held]),data['splits']['validation']).size
    assert not np.intersect1d(np.concatenate([train,held]),data['splits']['test']).size
    audit={'dataset':job['dataset'],'fold':job['fold'],'seed':job['seed'],'cohort':number,
       'training_participants':tr_people,'held_participants':people,'training_windows':len(train),
       'held_windows':len(held),'train_held_participant_overlap':0,'normalization_fit':'auxiliary training rows only',
       'checkpoint_selection':'fixed final epoch; no validation or held-source scoring',
       'source_indices_sha256':hashlib.sha256(train.tobytes()+held.tobytes()).hexdigest()}
    signature=json_signature({'study':STUDY_SIGNATURE,**audit})
    audit['signature']=signature
    split_path=folder/'source_split.json'
    if split_path.exists():assert json.loads(split_path.read_text())==audit
    else:save_json(split_path,audit)
    paths=[folder/(c+'.npz') for c in CONDITIONS]
    if all(cache_valid(p,signature) for p in paths):
        print(f'  auxiliary cohort {number}: reused all 10 conditions.',flush=True)
        return pack_from_caches(paths,signature),audit
    if REQUIRE_GPU and DEVICE.type!='cuda':
        raise RuntimeError('Auxiliary training/caches remain unfinished. Select Runtime > Change runtime type > T4 GPU, then rerun the same cell.')
    check_budget()
    raw=data['X'][train]
    mean=raw.mean(axis=(0,2),dtype=np.float64).astype(np.float32)
    std=np.maximum(raw.std(axis=(0,2),dtype=np.float64),1e-6).astype(np.float32)
    save_npz(folder/'normalization.npz',mean=mean,std=std,fit_source_indices=train,signature=np.asarray(signature))
    clean=np.ascontiguousarray((raw-mean[None,:,None])/std[None,:,None],dtype=np.float32)
    observed=np.ascontiguousarray((data['X'][held]-mean[None,:,None])/std[None,:,None],dtype=np.float32)
    groups=data['groups'];K=data['classes']
    stage_seed=job['seed']+zlib.crc32((job['dataset']+'|aux|'+str(number)).encode())%1000000
    print(f'  Auxiliary cohort {number}: train people {tr_people}; predict held people {people}.',flush=True)
    classifier=fit_auxiliary('classifier',clean,data['y'][train],groups,K,folder,signature,stage_seed)
    reconstructor=fit_auxiliary('reconstructor',clean,data['y'][train],groups,K,folder,signature,stage_seed+100)
    names=np.asarray(['keep']+[f'mask:{g["name"]}' for g in groups]+[f'reconstruct:{g["name"]}' for g in groups])
    for condition,path in zip(CONDITIONS,paths):
        if cache_valid(path,signature):continue
        check_budget()
        values=observed
        if condition!='clean':
            fault,level=condition.rsplit('_',1)
            rng=np.random.default_rng(stage_seed+zlib.crc32(condition.encode()))
            values=corrupt_signals(observed,groups,rng,fault,int(level)-1)
        p,r,s,seconds=action_outputs(classifier,reconstructor,values,groups,K)
        save_npz(path,signature=np.asarray(signature),condition=np.asarray(condition),split=np.asarray('oof'),
          y=data['y'][held],subjects=data['subjects'][held],window_ids=data['window_ids'][held],source_indices=held,
          action_names=names,probabilities=p,reconstruction_residuals=r,observed_group_statistics=s)
        save_json(path.with_name(path.name+'.sha256.json'),{'signature':signature,'sha256':file_sha256(path)})
        print(f'  held source {condition}: {len(held)} windows | {seconds:.1f}s',flush=True)
        del p,r,s,values
    del classifier,reconstructor,clean,observed,raw
    gc.collect()
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
    return pack_from_caches(paths,signature),audit


def combine_packs(packs):
    first=packs[0]
    for p in packs:
        assert p['conditions']==CONDITIONS and np.array_equal(p['action_names'],first['action_names'])
    out={k:np.concatenate([p[k] for p in packs]) for k in ['y','subjects','window_ids','source_indices']}
    out.update({k:np.concatenate([p[k] for p in packs],axis=1) for k in ['probabilities','residuals','statistics']})
    out.update(action_names=first['action_names'],conditions=CONDITIONS)
    assert len(np.unique(out['source_indices']))==len(out['y'])
    assert len(np.unique(out['window_ids']))==len(out['y'])
    return out


def subject_weights(subjects):
    people,inv,counts=np.unique(subjects,return_inverse=True,return_counts=True)
    return (1./counts[inv]).astype(np.float64)


def counterfactual_targets(predictions,y,subjects,classes):
    # Target is exact for ONE replaced prediction in a fixed keep confusion
    # matrix. It is a marginal utility surrogate, not additive/global F1.
    D,N,A=predictions.shape;J=A-1
    w=subject_weights(subjects);total=w.sum()
    weights=np.broadcast_to(w[None,:,None],(D,N,J)).copy()
    keep=predictions[:,:,0];actions=predictions[:,:,1:]
    correct=predictions==y[None,:,None]
    fix=(~correct[:,:,:1]&correct[:,:,1:]).astype(np.float32)
    harm=(correct[:,:,:1]&~correct[:,:,1:]).astype(np.float32)
    utility=np.zeros((D,N,J),dtype=np.float64)
    for d in range(D):
        C=np.bincount(y*classes+keep[d],weights=w,minlength=classes*classes).reshape(classes,classes)
        denominator=C.sum(0)+C.sum(1);tp=C.diagonal()
        old=np.divide(2.*tp,denominator,out=np.zeros(classes),where=denominator>0)
        b=np.broadcast_to(keep[d,:,None],(N,J));a=actions[d]
        wb=np.broadcast_to(w[:,None],(N,J))
        db=np.maximum(denominator[b]-wb,0.);da=denominator[a]+wb
        tpb=tp[b]-wb*(y[:,None]==b);tpa=tp[a]+wb*(y[:,None]==a)
        fb=np.divide(2.*tpb,db,out=np.zeros_like(db),where=db>1e-12)
        fa=np.divide(2.*tpa,da,out=np.zeros_like(da),where=da>1e-12)
        utility[d]=((fb-old[b])+(fa-old[a]))/classes*total/wb
        utility[d][a==b]=0.
    assert np.isfinite(utility).all()
    # Normalize sample weights to mean 1, preserving equal total mass per person.
    weights/=weights.mean()
    return utility.astype(np.float32),fix,harm,weights.astype(np.float32)


def fit_controller(name,X,target,weight,kind,run,signature,people):
    path=run/(name+'.joblib')
    if path.exists():
        payload=joblib.load(path)
        assert payload['signature']==signature and payload['training_participants']==people
        print('  controller',name,': reused.',flush=True)
        return payload['model']
    check_budget();start=time.perf_counter()
    if np.ptp(target)==0:
        model={'constant':float(target[0])}
    else:
        cls=HistGradientBoostingClassifier if kind=='binary' else HistGradientBoostingRegressor
        model=cls(learning_rate=.05,max_iter=MAX_ITER,max_leaf_nodes=15,max_depth=4,
            min_samples_leaf=40,l2_regularization=5.,max_bins=64,categorical_features=None,
            early_stopping=False,random_state=2026+zlib.crc32(name.encode())%10000)
        with threadpool_limits(limits=2):model.fit(X,target,sample_weight=weight)
    elapsed=time.perf_counter()-start
    atomic_joblib(path,{'signature':signature,'model':model,'kind':kind,'training_rows':len(target),
        'training_participants':people,'training_seconds':elapsed})
    print(f'  controller {name}: {len(target)} rows, {len(people)} people | {elapsed:.1f}s',flush=True)
    return model


def head_prediction(model,X,kind):
    if isinstance(model,dict):return np.full(len(X),model['constant'],dtype=np.float32)
    with threadpool_limits(limits=2):
        if kind=='binary':return model.predict_proba(X)[:,list(model.classes_).index(1)].astype(np.float32)
        return model.predict(X).astype(np.float32)


def scored_actions(scores,threshold):
    if threshold is None:return np.zeros(scores.shape[:2],dtype=np.int64)
    return np.where(scores.max(-1)>threshold,scores.argmax(-1)+1,0)


def select_family(pack,scores_by_penalty,family):
    base=aggregate_rows(evaluate_choices(pack,np.zeros(scores_by_penalty[1.].shape[:2],dtype=np.int64),'keep'))
    best=None;trials=[]
    for penalty in HARM_PENALTIES:
        for threshold in THRESHOLDS:
            actions=scored_actions(scores_by_penalty[penalty],threshold)
            summary=aggregate_rows(evaluate_choices(pack,actions,family))
            feasible=(summary['clean_macro_f1']>=base['clean_macro_f1']-CLEAN_F1_TOLERANCE-1e-12
                and summary['clean_harmful_rate']<=MAX_HARM+1e-12 and summary['harmful_rate']<=MAX_HARM+1e-12)
            trials.append({'family':family,'harm_penalty':penalty,'threshold':threshold,'feasible':feasible,**summary})
            rank=(summary['macro_f1'],summary['clean_macro_f1'],-summary['harmful_rate'],-summary['intervention_rate'],threshold is None)
            if feasible and (best is None or rank>best['rank']):
                best={'rank':rank,'settings':{'family':family,'harm_penalty':penalty,'threshold':threshold},
                      'summary':summary,'actions':actions,'structural_keep_only':threshold is None}
    assert best is not None
    del best['rank']
    return best,trials


def process_job(job):
    run=OUTPUT/'runs'/identity(job);run.mkdir(parents=True,exist_ok=True)
    done=run/'development_complete.json'
    if done.exists():
        result=json.loads(done.read_text());assert result['study_signature']==STUDY_SIGNATURE
        for a in result['artifacts']:assert file_sha256(run/a['filename'])==a['sha256']
        print(identity(job),': reused completed pilot.',flush=True)
        return result
    print('\nRUN',identity(job),flush=True)
    data=load_source_data(job)
    original=load_original_pack(job,'policy');validation=load_original_pack(job,'validation')
    for pack,split in [(original,'policy'),(validation,'validation')]:
        assert np.array_equal(pack['source_indices'],data['splits'][split])
        assert np.array_equal(pack['y'],data['y'][pack['source_indices']])
        assert np.array_equal(pack['subjects'],data['subjects'][pack['source_indices']])
        assert np.array_equal(pack['window_ids'],data['window_ids'][pack['source_indices']])
    packs=[original];audits=[]
    for i,people in enumerate(data['blocks']):
        pack,audit=auxiliary_cohort(job,data,i,people,run)
        packs.append(pack);audits.append(audit)
    pooled=combine_packs(packs)
    assert np.array_equal(np.sort(pooled['source_indices']),np.sort(data['pool']))
    assert not np.intersect1d(pooled['subjects'],validation['subjects']).size
    people=np.unique(pooled['subjects']).tolist()
    original_people=np.unique(original['subjects']).tolist()
    audits.insert(0,{'dataset':job['dataset'],'fold':job['fold'],'seed':job['seed'],'cohort':'original_policy',
      'training_participants':np.unique(data['subjects'][data['splits']['train']]).tolist(),
      'held_participants':original_people,'training_windows':len(data['splits']['train']),
      'held_windows':len(original['y']),'train_held_participant_overlap':0,
      'normalization_fit':'original train only','checkpoint_selection':'original frozen validation-selected checkpoint'})
    save_json(run/'source_audit.json',audits)
    scale=np.maximum(np.quantile(original['residuals'][0],.95,axis=0),1e-8).astype(np.float32)
    X,names=policy_features(pooled['probabilities'],pooled['residuals'],pooled['statistics'],scale)
    VX,vnames=policy_features(validation['probabilities'],validation['residuals'],validation['statistics'],scale)
    assert names==vnames
    predictions=pooled['probabilities'].argmax(-1)
    target,fix,harm,weight=counterfactual_targets(predictions,pooled['y'],pooled['subjects'],data['classes'])
    D,N,J=target.shape;VD,VN,VA,VK=validation['probabilities'].shape
    signature=json_signature({'study':STUDY_SIGNATURE,'job':job,'source_people':people,'features':names})
    settings={'signature':signature,'feature_names':names,'residual_scale':scale.tolist(),
      'source_participants':people,'validation_participants':np.unique(validation['subjects']).tolist(),
      'source_windows':N,'source_views':D*N,'action_rows':X.shape[0],
      'wrong_to_wrong_nonzero_f1_targets':int(np.sum((fix==0)&(harm==0)&(target!=0))),
      'f1_target_min':float(target.min()),'f1_target_max':float(target.max()),
      'excluded_runtime_features':['labels','participants','window_ids','split','condition','fault location','severity'],
      'source_jackknife_limitation':plan['jackknife_limitation']}
    save_json(run/'controller_inputs.json',settings)
    print(f'  Controller pool: {len(original_people)} -> {len(people)} people | {N} original windows | {X.shape[0]} action rows',flush=True)
    print('  Wrong-to-wrong cases with nonzero F1 targets:',settings['wrong_to_wrong_nonzero_f1_targets'],flush=True)
    def train_head(name,t,kind,mask=None,subset_people=None):
        if mask is None:return fit_controller(name,X,t.reshape(-1),weight.reshape(-1),kind,run,signature,people)
        rows=np.broadcast_to(mask[None,:,None],(D,N,J)).reshape(-1)
        return fit_controller(name,X[rows],t.reshape(-1),subject_weight_for_mask(mask),kind,run,signature,subset_people)
    def subject_weight_for_mask(mask):
        w=subject_weights(pooled['subjects'][mask]);w/=w.mean()
        return np.broadcast_to(w[None,:,None],(D,int(mask.sum()),J)).reshape(-1).copy().astype(np.float32)
    hf=train_head('pooled_fix',fix,'binary');hh=train_head('pooled_harm',harm,'binary')
    hu=train_head('pooled_f1',target,'regression')
    shape=(VD,VN,J)
    pf=head_prediction(hf,VX,'binary').reshape(shape);ph=head_prediction(hh,VX,'binary').reshape(shape)
    pu=head_prediction(hu,VX,'regression').reshape(shape)
    del hf,hh,hu,target,fix,harm,weight
    source_blocks=np.array_split(np.random.default_rng(2026).permutation(people),3)
    source_utilities=[];source_harms=[];jackknife_audit=[]
    for b,block in enumerate(source_blocks):
        mask=~np.isin(pooled['subjects'],block)
        subset_people=np.unique(pooled['subjects'][mask]).tolist()
        t,_,h,_=counterfactual_targets(predictions[:,mask],pooled['y'][mask],pooled['subjects'][mask],data['classes'])
        a=train_head(f'jackknife_{b}_f1',t,'regression',mask,subset_people)
        hmodel=train_head(f'jackknife_{b}_harm',h,'binary',mask,subset_people)
        source_utilities.append(head_prediction(a,VX,'regression').reshape(shape))
        source_harms.append(head_prediction(hmodel,VX,'binary').reshape(shape))
        jackknife_audit.append({'block':b,'excluded_people':block.tolist(),'training_people':subset_people,
            'targets_recomputed_without_excluded_people':True,'training_rows':int(mask.sum()*D*J)})
        del a,hmodel,t,h
    save_json(run/'jackknife_audit.json',jackknife_audit)
    su=np.stack(source_utilities);sh=np.stack(source_harms)
    score_families={f:{} for f in FAMILIES}
    for penalty in HARM_PENALTIES:
        score_families['oof_accuracy_utility'][penalty]=pf-penalty*ph
        score_families['oof_f1_utility'][penalty]=pu-(penalty-1.)*ph
        source_score=su-(penalty-1.)*sh
        score_families['source_mean_f1'][penalty]=source_score.mean(0)
        score_families['source_min_f1'][penalty]=source_score.min(0)
    rows=[];trials=[];selected={};decisions=[]
    for family in FAMILIES:
        best,grid=select_family(validation,score_families[family],family)
        decisions.append(best.pop('actions'));selected[family]=best
        rows.append({'dataset':job['dataset'],'fold':job['fold'],'seed':job['seed'],'method':family,
                     'source':'new validation selection','structural_keep_only':best['structural_keep_only'],**best['summary']})
        trials.extend(grid)
        print('  Selected',family,':',best['settings'],'| val F1',round(best['summary']['macro_f1']*100,3),flush=True)
    # Reproduce simple baselines on identical frozen validation candidates.
    vp=validation['probabilities'];ratio=validation['residuals']/scale[None,None,:]
    suspect=ratio.argmax(-1);gate=ratio.max(-1)>1.
    baselines={'keep':np.zeros((VD,VN),dtype=np.int64),'confidence_select':vp.max(-1).argmax(-1),
      'residual_gate_mask':np.where(gate,1+suspect,0),
      'residual_gate_reconstruct':np.where(gate,1+len(scale)+suspect,0)}
    for name,choices in baselines.items():
        rows.append({'dataset':job['dataset'],'fold':job['fold'],'seed':job['seed'],'method':name,
            'source':'recomputed frozen validation cache','structural_keep_only':name=='keep',
            **aggregate_rows(evaluate_choices(validation,choices,name))})
    # Prior validation scores are shown as recorded comparisons, not refitted
    # controls. This explicitly retains stronger controllers from Notebooks 7/9.
    for number,filename,key,summary_key in [(7,'notebook07_run_report.json','selected_controls','summary'),
                                           (9,'notebook09_run_report.json','selected_methods','summary')]:
        report=PRIOR_REPORTS[number]
        r=next(r for r in report['completed_results'] if identity(r)==identity(job))
        for name,item in r[key].items():
            rows.append({'dataset':job['dataset'],'fold':job['fold'],'seed':job['seed'],'method':name,
               'source':f'recorded Notebook {number:02d} validation selection'+(' (no harm guard)' if name.endswith('_accuracy') else ''),
               'structural_keep_only':item.get('structural_keep_only',item['settings'].get('threshold',0)==2.),**item[summary_key]})
    oldroot=PROJECT/'outputs/notebook05_v1'
    oldresult=next(r for r in PRIOR_REPORTS[5]['completed_results'] if identity(r)==identity(job))
    inv={r['filename']:r for r in oldresult['artifacts']}
    oldname='runs/'+identity(job)+'/selected_policy.json'
    oldpath=verified_copy(oldroot/oldname,'nb05/'+oldname,inv['selected_policy.json']['sha256'])
    old=json.loads(oldpath.read_text())['selected_methods']['harm_aware_policy']
    rows.append({'dataset':job['dataset'],'fold':job['fold'],'seed':job['seed'],'method':'harm_aware_policy',
       'source':'recorded Notebook 05 validation selection','structural_keep_only':old['settings']['threshold']==2.,
       **old['validation_summary']})
    save_csv(run/'validation_summary.csv',pd.DataFrame(rows))
    save_csv(run/'validation_trials.csv',pd.DataFrame(trials))
    save_json(run/'selected_policies.json',{'signature':signature,'selected_methods':selected})
    save_npz(run/'validation_decisions.npz',signature=np.asarray(signature),methods=np.asarray(FAMILIES),
       actions=np.stack(decisions),conditions=np.asarray(CONDITIONS),y=validation['y'],subjects=validation['subjects'],
       source_indices=validation['source_indices'],window_ids=validation['window_ids'])
    artifacts=[{'filename':p.name,'sha256':file_sha256(p)} for p in run.iterdir() if p.is_file() and p.name!='development_complete.json']
    result={**job,'study_signature':STUDY_SIGNATURE,'source_participants':people,'original_policy_participants':original_people,
       'validation_participants':np.unique(validation['subjects']).tolist(),'source_windows':N,'action_rows':X.shape[0],
       'auxiliary_pairs':len(data['blocks']),'selected_methods':selected,'test_predictions_read_or_created':0,
       'source_audit':audits,'jackknife_audit':jackknife_audit,'artifacts':artifacts}
    save_json(done,result)
    del data,pooled,original,validation,X,VX,predictions
    gc.collect()
    return result


completed=[];paused=None
try:
    for job in JOBS:
        completed.append(process_job(job))
except PauseStudy as e:
    paused=str(e)
    print('\nPAUSED:',paused,'Rerun the same cell to continue.',flush=True)

report={'code_version':CODE_VERSION,'study_signature':STUDY_SIGNATURE,'study_complete':len(completed)==len(JOBS),
 'configurations_complete':len(completed),'configurations_required':len(JOBS),'completed_results':completed,
 'pause_reason':paused,'new_test_inference':0,'test_predictions_read':0,'held_out_stuck_evaluations':0,
 'original_models_or_outputs_modified':False,'all_results_are_development_validation':True,
 'harm_guarantee':False,'novelty_or_improvement_guarantee':False,'jackknife_limitation':plan['jackknife_limitation'],
 'next':'Review both validation pilots and stronger recorded controls before any further evaluation. Old tests remain exploratory.'}
save_json(OUTPUT/'notebook11_run_report.json',report)
tables=[];audit_rows=[]
for r in completed:
    tables.append(pd.read_csv(OUTPUT/'runs'/identity(r)/'validation_summary.csv'))
    for a in r['source_audit']:
        row=dict(a)
        for k in ['training_participants','held_participants']:row[k]=json.dumps(row[k])
        audit_rows.append(row)
if tables:
    summary=pd.concat(tables,ignore_index=True)
    save_csv(OUTPUT/'notebook11_validation_summary.csv',summary)
    save_csv(OUTPUT/'notebook11_source_audit.csv',pd.DataFrame(audit_rows))
    display=summary[['dataset','method','macro_f1','clean_macro_f1','harmful_rate','clean_harmful_rate','structural_keep_only']].copy()
    for c in ['macro_f1','clean_macro_f1','harmful_rate','clean_harmful_rate']:display[c]=(display[c]*100).round(3)
    print('\nVALIDATION PILOT ONLY | percentages | one fold and one seed',flush=True)
    print(display.to_string(index=False),flush=True)
print('\nNOTEBOOK 11', 'COMPLETE' if report['study_complete'] else 'PAUSED',flush=True)
print(f'Completed: {len(completed)}/{len(JOBS)} | New final test inference: 0 | Stuck faults evaluated: 0',flush=True)
print('Validation guards are empirical; minimum source utility is a heuristic, not a risk guarantee.',flush=True)
print('Outputs:',OUTPUT,flush=True)
print('Share final output, notebook11_validation_summary.csv, notebook11_run_report.json, notebook11_source_audit.csv.',flush=True)
inventory=[{'filename':p.relative_to(OUTPUT).as_posix(),'sha256':file_sha256(p),'bytes':p.stat().st_size}
           for p in sorted(OUTPUT.iterdir()) if p.is_file() and p.name!='notebook11_artifact_inventory.json']
save_json(OUTPUT/'notebook11_artifact_inventory.json',inventory)


Mounted at /content/drive
NOTEBOOK 11 | cross-fitted F1 repair with source disagreement rejection | one executable cell
Development pilot: both datasets, fold_0, seed 2026. No final test predictions or stuck-fault evaluation.
Device: Tesla T4 | PyTorch: 2.11.0+cu130
Auxiliary models train from scratch; each held source cohort is excluded from fitting and normalization.
Development plan frozen: 678d54cf4f7b91c0

RUN dsads_fold_0_seed_2026
  Auxiliary cohort 0: train people [3, 4, 7, 8]; predict held people [6].
  classifier epoch 01/30 | training loss 1.3183 | no validation selection
  classifier epoch 05/30 | training loss 0.1676 | no validation selection
  classifier epoch 10/30 | training loss 0.1273 | no validation selection
  classifier epoch 15/30 | training loss 0.0788 | no validation selection
  classifier epoch 20/30 | training loss 0.0627 | no validation selection
  classifier epoch 25/30 | training loss 0.0466 | no validation selection
  classifier epoch 30/30 | training loss